In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error

df = pd.read_csv("car_fuel_efficiency_2026.csv")

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [2]:
columns = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
    "fuel_efficiency_mpg"
]

df = df[columns]

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

Dataset shape: (10000, 5)

Missing values:
engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


In [3]:
df["horsepower"].median()

np.float64(254.0)

In [4]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

(6000, 5)
(2000, 5)
(2000, 5)


In [5]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year"
]

target = "fuel_efficiency_mpg"

X_train = df_train[features].copy()
y_train = df_train[target].values

X_val = df_val[features].copy()
y_val = df_val[target].values


In [6]:
X_train_0 = X_train.fillna(0)
X_val_0 = X_val.fillna(0)

model_0 = LinearRegression()
model_0.fit(X_train_0, y_train)

y_pred_0 = model_0.predict(X_val_0)

rmse_0 = np.sqrt(mean_squared_error(y_val, y_pred_0))

print("RMSE with 0:", round(rmse_0, 3))

RMSE with 0: 2.205


In [7]:
mean_hp = X_train["horsepower"].mean()

X_train_mean = X_train.fillna(mean_hp)
X_val_mean = X_val.fillna(mean_hp)

model_mean = LinearRegression()
model_mean.fit(X_train_mean, y_train)

y_pred_mean = model_mean.predict(X_val_mean)

rmse_mean = np.sqrt(mean_squared_error(y_val, y_pred_mean))

print("RMSE with mean:", round(rmse_mean, 3))

RMSE with mean: 2.202


In [8]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    if r == 0:
        model = LinearRegression()
    else:
        model = Ridge(alpha=r)

    X_train_0 = X_train.fillna(0)
    X_val_0 = X_val.fillna(0)

    model.fit(X_train_0, y_train)
    y_pred = model.predict(X_val_0)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(r, round(rmse, 4))

0 2.2053
0.01 2.2053
0.1 2.2053
1 2.2053
5 2.2053
10 2.2053
100 2.2053


In [9]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    if r == 0:
        model = LinearRegression()
    else:
        model = Ridge(alpha=r)

    X_train_0 = X_train.fillna(0)
    X_val_0 = X_val.fillna(0)

    model.fit(X_train_0, y_train)
    y_pred = model.predict(X_val_0)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(r, f"{rmse:.10f}")

0 2.2052931948
0.01 2.2052931948
0.1 2.2052931952
1 2.2052931993
5 2.2052932176
10 2.2052932404
100 2.2052936541


In [10]:
scores = []

for seed in range(10):
    np.random.seed(seed)
    idx = np.arange(len(df))
    np.random.shuffle(idx)

    df_train_seed = df.iloc[idx[:n_train]].copy()
    df_val_seed = df.iloc[idx[n_train:n_train + n_val]].copy()

    X_train_seed = df_train_seed[features].fillna(0)
    y_train_seed = df_train_seed[target].values

    X_val_seed = df_val_seed[features].fillna(0)
    y_val_seed = df_val_seed[target].values

    model = LinearRegression()
    model.fit(X_train_seed, y_train_seed)

    y_pred = model.predict(X_val_seed)

    rmse = np.sqrt(mean_squared_error(y_val_seed, y_pred))
    scores.append(rmse)

    print(seed, round(rmse, 3))

print("Standard deviation:", round(np.std(scores), 3))

0 2.239
1 2.202
2 2.163
3 2.204
4 2.202
5 2.246
6 2.27
7 2.191
8 2.217
9 2.207
Standard deviation: 0.029


In [11]:
print("Standard deviation:", round(np.std(scores), 3))

Standard deviation: 0.029


In [13]:
df.shape

(10000, 5)

In [14]:
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [15]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

len(df_train), len(df_val), len(df_test)


(6000, 2000, 2000)

In [16]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year"
]

y_train = df_train["fuel_efficiency_mpg"].values
y_val = df_val["fuel_efficiency_mpg"].values
y_test = df_test["fuel_efficiency_mpg"].values

X_train = df_train[features].copy()
X_val = df_val[features].copy()
X_test = df_test[features].copy()

In [17]:
median_hp = X_train["horsepower"].median()

X_train["horsepower"] = X_train["horsepower"].fillna(median_hp)
X_val["horsepower"] = X_val["horsepower"].fillna(median_hp)
X_test["horsepower"] = X_test["horsepower"].fillna(median_hp)

X_train.isnull().sum()

engine_displacement    0
horsepower             0
vehicle_weight         0
model_year             0
dtype: int64

In [18]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [19]:
import numpy as np

y_pred = model.predict(X_test)

rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

print("Test RMSE:", rmse)
print("Rounded:", round(rmse, 3))

Test RMSE: 2.130974258395652
Rounded: 2.131


In [20]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=9
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=9
)

In [21]:
import pandas as pd

df_train_full = pd.concat(
    [df_train, df_val], ignore_index=True
)

len(df_train_full), len(df_test)

(8000, 2000)

In [22]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year"
]

X_train_full = df_train_full[features].fillna(0)
X_test = df_test[features].fillna(0)

y_train_full = df_train_full["fuel_efficiency_mpg"].values
y_test = df_test["fuel_efficiency_mpg"].values

In [23]:
from sklearn.linear_model import Ridge
import numpy as np

model = Ridge(alpha=0.001)
model.fit(X_train_full, y_train_full)

y_pred = model.predict(X_test)

rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

print("RMSE:", rmse)
print("Rounded RMSE:", round(rmse, 3))

RMSE: 2.225350036581325
Rounded RMSE: 2.225


In [24]:
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

In [25]:
print(df.shape)
print(len(df_train_full), len(df_test))
print(rmse)

(10000, 5)
8000 2000
2.225350036581325


In [26]:
print(model)
print(model.alpha)
print(X_train_full.isnull().sum())

Ridge(alpha=0.001)
0.001
engine_displacement    0
horsepower             0
vehicle_weight         0
model_year             0
dtype: int64
